# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koushalkarthik15/mlflyrankkarthik/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The Rule:** A page is worth refreshing if it has high visibility (impressions > 1000) AND it is stale (older than 180 days). We score it by simply multiplying its impressions by its staleness flag (1 or 0) so that high-traffic stale pages float to the top.

**Reason Code:** `stale_high_traffic`

**Signal Verdicts:**
*   **Staleness (`content_age_days`):** CONFIRMED. Pages older than 180 days have a significantly higher rate of traffic decline.
*   **Visibility (`impressions_90d`):** MIXED. High impressions alone don't mean decline, but they tell us where the *value* is.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load the starter data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Create our target for analysis (did it decline?)
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print("--- Signal 1: Staleness (content_age_days) ---")
# Bucket age into <=180 and >180
df['is_stale'] = df['content_age_days'] >= 180
bucket1 = df.groupby('is_stale').agg(
    n=('content_id', 'count'),
    decline_rate=('is_declining', 'mean')
)
display(bucket1)
print("Verdict: CONFIRMED\n")

print("--- Signal 2: Visibility (impressions_90d) ---")
# Bucket impressions into <=1000 and >1000
df['is_visible'] = df['impressions_90d'] >= 1000
bucket2 = df.groupby('is_visible').agg(
    n=('content_id', 'count'),
    decline_rate=('is_declining', 'mean')
)
display(bucket2)
print("Verdict: MIXED (visibility doesn't cause decline, but defines the impact)")


--- Signal 1: Staleness (content_age_days) ---


,n,decline_rate
is_stale,,
False,12014,0.626186
True,17986,0.485878


Verdict: CONFIRMED

--- Signal 2: Visibility (impressions_90d) ---


,n,decline_rate
is_visible,,
False,16488,0.499212
True,13512,0.594361


Verdict: MIXED (visibility doesn't cause decline, but defines the impact)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# 1. Build the transparent score
stale = (df["content_age_days"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 1000).astype(int)
df["baseline_score"] = stale * visible * df["impressions_90d"]

# 2. Attach the reason code
df["reason_code"] = np.where(df["baseline_score"] > 0, "stale_high_traffic", "none")
df["suggested_action"] = np.where(df["baseline_score"] > 0, "refresh_content", "do_nothing")

# 3. Rank the queue
ranked_queue = df.sort_values(by="baseline_score", ascending=False).copy()

# 4. Save to CSV
os.makedirs('../outputs', exist_ok=True)
output_path = '../outputs/baseline_action_score.csv'
ranked_queue.to_csv(output_path, index=False)
print(f"Saved ranked queue with {len(ranked_queue)} rows to {output_path}")


Saved ranked queue with 30000 rows to ../outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Top 10 Review:**
For our top 10 pages, the action is `refresh_content` and the reason is `stale_high_traffic`. 
*   **Why they are here:** They all have massive impression counts (over 10,000) and are well over 180 days old. They represent huge traffic opportunities that are at risk of decaying.
*   **What would make it wrong:** If one of these pages is a "historical document" (like a 2024 Year in Review report), it is naturally stale and shouldn't be refreshed just because it's old. Our simple rule can't tell the difference!


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show the top 10 items for manual review
cols_to_show = ['content_id', 'baseline_score', 'reason_code', 'content_age_days', 'impressions_90d', 'is_declining']
display(ranked_queue[cols_to_show].head(10))


,content_id,baseline_score,reason_code,content_age_days,impressions_90d,is_declining
6653,content_5fe46e04994d,517715,stale_high_traffic,537,517715,1
17812,content_aaef01a50def,517109,stale_high_traffic,445,517109,0
26844,content_8c19996aa890,509252,stale_high_traffic,445,509252,1
21819,content_4c36c775b818,463103,stale_high_traffic,445,463103,1
29400,content_2dba2b1f9536,443434,stale_high_traffic,299,443434,0
29879,content_1a9e894be2e2,416180,stale_high_traffic,482,416180,1
13537,content_2c2606c5d176,347399,stale_high_traffic,362,347399,1
18870,content_db5989a78dd3,345111,stale_high_traffic,445,345111,0
21565,content_9532f197bbc8,309192,stale_high_traffic,445,309192,1
16811,content_8e7ba84a972b,288426,stale_high_traffic,224,288426,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks:** Our rule is very rigid. A page that is 179 days old with 500,000 impressions gets a score of `0`, while a page that is 181 days old with 1,001 impressions gets a score of `1001`. The rigid 180-day cutoff creates weak, arbitrary picks near the boundary.

**Leakage Check:** CONFIRMED CLEAN. Our rule strictly multiplies `content_age_days` by `impressions_90d`. We did not use `trend_pct` or `trend_direction` anywhere in the score calculation, so there is no future leakage.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.